# Paper Results: Survey x Simulation

This notebook mixes the two sides of the project:

- `survey_results.ipynb` explains and explores the human survey data.
- `simulation_results.ipynb` explains and explores the logged simulation benchmark data.
- This file checks whether the simulation benchmark agrees with the survey outcomes.

The main question here is not whether the scores are numerically identical. The survey is a human rating on a `-3..+3` Likert-style scale, while the simulation benchmark is a normalized objective score on `0..1`. The question is whether they tell the same story when we compare the same policy/POV pairs.

## Decision Logic

We will treat this as an alignment check:

1. Import and clean the survey answers.
2. Import and aggregate the simulation metrics.
3. Put both datasets on comparable condition/axis labels.
4. Compare rankings and paired differences, not raw absolute scores.
5. Mark where the simulation and survey agree, disagree, or are too close to call.

Useful references for the methods used here:

- Exploratory data analysis: [NIST/SEMATECH EDA handbook](https://www.nist.gov/publications/nistsematech-e-handbook-statistical-methods-chapter-1-exploratory-data-analysis)
- Bootstrap confidence intervals: [SciPy `stats.bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html)
- Rank correlation / Spearman idea: [SciPy `stats.spearmanr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)

We implement the bootstrap and rank correlation directly with `numpy`/`pandas` so this notebook does not require SciPy.

## Imports And Shared Mappings

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

SURVEY_CSV = Path("survey_results") / "Human-Robot Interaction Evaluation (Responses) - Form Responses 1.csv"
SIMULATION_ROOT = Path("artifacts/local-validation")

RANDOM_SEED = 20260514

In [ ]:
Q_MAPPING = {
    "q1": {"policy": "A", "pov": "first_person", "condition_label": "Policy A - first person"},
    "q2": {"policy": "B", "pov": "first_person", "condition_label": "Policy B - first person"},
    "q3": {"policy": "A", "pov": "third_person", "condition_label": "Policy A - third person"},
    "q4": {"policy": "B", "pov": "third_person", "condition_label": "Policy B - third person"},
}

SURVEY_QUESTION_AXIS = {
    "1": "competence_control",
    "2": "safety",
    "3": "awareness",
    "4": "positive_impression",
}

SIMULATION_AXIS_MAP = {
    "perceived_dexterity": "competence_control",
    "perceived_safety": "safety",
    "perceived_social_awareness": "awareness",
    "impression": "positive_impression",
}

EPISODE_LABELS = {
    "ep1": "no_obstacles",
    "ep2": "fixed_obstacles",
    "ep3": "moving_people",
}

AXES = ["competence_control", "safety", "awareness", "positive_impression"]
AXES_WITH_GLOBAL = ["global"] + AXES

COMPARISONS = [
    ("q1", "q2", "A vs B - first-person POV", "Policy A minus Policy B, first-person videos"),
    ("q3", "q4", "A vs B - third-person POV", "Policy A minus Policy B, third-person videos"),
    ("q1", "q3", "First vs third POV - Policy A", "First-person minus third-person for Policy A"),
    ("q2", "q4", "First vs third POV - Policy B", "First-person minus third-person for Policy B"),
]

## Import Survey Results

The survey data is participant-level. Every participant saw all four videos, so policy and POV comparisons are paired within participant. We keep that paired structure when computing differences.

In [ ]:
def convert_answer_to_number(answer):
    answer_text = str(answer)
    for score_text, score_number in {"-3": -3, "-2": -2, "-1": -1, "0": 0, "+1": 1, "+2": 2, "+3": 3}.items():
        if answer_text.startswith(score_text):
            return score_number
    return np.nan


def familiarity_to_boolean(answer):
    answer_text = str(answer)
    if answer_text == "True" or answer_text.startswith("Daily interaction"):
        return True
    if answer_text == "False" or answer_text.startswith("No daily contact"):
        return False
    return pd.NA


raw_survey = pd.read_csv(SURVEY_CSV)

rename_columns = {}
for old_column_name in raw_survey.columns:
    if old_column_name == "Timestamp":
        rename_columns[old_column_name] = "timestamp"
    elif old_column_name.startswith("Background / Familiarity"):
        rename_columns[old_column_name] = "robotics_familiarity"
    elif old_column_name.startswith("Q"):
        question_code = old_column_name.split(" ", 1)[0]
        rename_columns[old_column_name] = question_code.lower().replace(".", "_")

survey_wide = raw_survey.rename(columns=rename_columns).copy()
survey_wide.insert(0, "participant_id", np.arange(1, len(survey_wide) + 1))

score_columns = [column for column in survey_wide.columns if re.fullmatch(r"q[1-4]_[1-4]", column)]
for column in score_columns:
    survey_wide[column] = survey_wide[column].apply(convert_answer_to_number)

survey_wide["robotics_familiarity"] = survey_wide["robotics_familiarity"].apply(familiarity_to_boolean).astype("boolean")

survey_rows = []
for condition, metadata in Q_MAPPING.items():
    for question_number, axis in SURVEY_QUESTION_AXIS.items():
        column = f"{condition}_{question_number}"
        if column not in survey_wide.columns:
            continue
        for _, row in survey_wide.iterrows():
            survey_rows.append(
                {
                    "participant_id": row["participant_id"],
                    "robotics_familiarity": row["robotics_familiarity"],
                    "condition": condition,
                    "condition_label": metadata["condition_label"],
                    "policy": metadata["policy"],
                    "pov": metadata["pov"],
                    "axis": axis,
                    "score": row[column],
                }
            )

survey_long = pd.DataFrame(survey_rows)
survey_long["score_norm"] = (survey_long["score"] + 3) / 6

survey_global = (
    survey_long.groupby(["participant_id", "robotics_familiarity", "condition", "condition_label", "policy", "pov"], dropna=False)
    .agg(score=("score", "mean"), score_norm=("score_norm", "mean"))
    .reset_index()
)
survey_global["axis"] = "global"

survey_analysis_long = pd.concat([survey_long, survey_global], ignore_index=True, sort=False)

print(f"Survey participants: {survey_wide['participant_id'].nunique()}")
print(f"Survey long rows including global: {len(survey_analysis_long)}")
display(survey_analysis_long.head())

In [ ]:
survey_keys = ["condition", "condition_label", "policy", "pov", "axis"]

survey_summary = (
    survey_analysis_long.groupby(survey_keys, dropna=False)
    .agg(
        n=("score", "count"),
        survey_mean_raw=("score", "mean"),
        survey_median_raw=("score", "median"),
        survey_mean_0_1=("score_norm", "mean"),
        survey_median_0_1=("score_norm", "median"),
        survey_std_0_1=("score_norm", "std"),
    )
    .reset_index()
)

survey_polarity = (
    survey_analysis_long.assign(
        negative=survey_analysis_long["score"] < 0,
        neutral=survey_analysis_long["score"] == 0,
        positive=survey_analysis_long["score"] > 0,
    )
    .groupby(survey_keys, dropna=False)[["negative", "neutral", "positive"]]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"negative": "percent_negative", "neutral": "percent_neutral", "positive": "percent_positive"})
)

survey_summary = survey_summary.merge(survey_polarity, on=survey_keys, how="left")

display(survey_summary.sort_values(["axis", "condition"]))

## Import Simulation Results

The simulation data is episode-level. It has repeated logged runs, not human participants. We aggregate repeated runs within each `condition x episode x axis`, then summarize across the three episode slots.

In [ ]:
def metadata_from_csv_path(path: Path) -> dict:
    relative_parts = path.relative_to(SIMULATION_ROOT).parts
    condition = relative_parts[0] if relative_parts else None
    run_label = relative_parts[1] if len(relative_parts) > 1 else path.parent.name
    episode_match = re.search(r"ep(\d+)", run_label)
    episode_slot = f"ep{episode_match.group(1)}" if episode_match else None
    metadata = Q_MAPPING.get(condition, {})
    return {
        "condition": condition,
        "condition_label": metadata.get("condition_label"),
        "policy": metadata.get("policy"),
        "pov": metadata.get("pov"),
        "episode_slot": episode_slot,
        "episode_type": EPISODE_LABELS.get(episode_slot),
        "run_label": run_label,
        "csv_path": str(path),
    }


simulation_csv_paths = sorted(SIMULATION_ROOT.rglob("episode-metrics-*.csv"))
if not simulation_csv_paths:
    raise FileNotFoundError(f"No simulation CSV files found under {SIMULATION_ROOT.resolve()}")

simulation_frames = []
for csv_path in simulation_csv_paths:
    frame = pd.read_csv(csv_path)
    for key, value in metadata_from_csv_path(csv_path).items():
        frame[key] = value
    simulation_frames.append(frame)

simulation_raw = pd.concat(simulation_frames, ignore_index=True)

for column in SIMULATION_AXIS_MAP:
    simulation_raw[column] = pd.to_numeric(simulation_raw[column], errors="coerce")

if "technical_valid" in simulation_raw.columns:
    simulation_raw["technical_valid_bool"] = simulation_raw["technical_valid"].astype(str).str.lower().eq("true")

simulation_episode_axes = (
    simulation_raw.groupby(["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"], dropna=False)[list(SIMULATION_AXIS_MAP)]
    .mean()
    .reset_index()
    .rename(columns=SIMULATION_AXIS_MAP)
)

simulation_axis_long = simulation_episode_axes.melt(
    id_vars=["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"],
    value_vars=AXES,
    var_name="axis",
    value_name="sim_score",
).dropna(subset=["sim_score"])

simulation_global = (
    simulation_axis_long.groupby(["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"], dropna=False)
    .agg(sim_score=("sim_score", "mean"))
    .reset_index()
)
simulation_global["axis"] = "global"

simulation_analysis_long = pd.concat([simulation_axis_long, simulation_global], ignore_index=True, sort=False)

print(f"Simulation rows: {len(simulation_raw)}")
print(f"Simulation CSV files: {len(simulation_csv_paths)}")
display(simulation_analysis_long.head())

In [ ]:
simulation_keys = ["condition", "condition_label", "policy", "pov", "axis"]

simulation_summary = (
    simulation_analysis_long.groupby(simulation_keys, dropna=False)
    .agg(
        sim_mean_0_1=("sim_score", "mean"),
        sim_median_0_1=("sim_score", "median"),
        sim_std_0_1=("sim_score", "std"),
        n_episode_values=("sim_score", "count"),
        n_episode_slots=("episode_slot", "nunique"),
    )
    .reset_index()
)

simulation_run_quality = (
    simulation_raw.groupby(["condition", "condition_label", "policy", "pov"], dropna=False)
    .agg(
        n_logged_runs=("csv_path", "count"),
        technical_valid_rate=("technical_valid_bool", "mean") if "technical_valid_bool" in simulation_raw.columns else ("csv_path", "count"),
    )
    .reset_index()
)

simulation_summary = simulation_summary.merge(simulation_run_quality, on=["condition", "condition_label", "policy", "pov"], how="left")

display(simulation_summary.sort_values(["axis", "condition"]))

## Put Survey And Simulation On The Same Table

The survey score is converted to `0..1` only for comparison. A value of `0.5` means neutral on the original `-3..+3` scale. The simulation score already lives on `0..1`.

In [ ]:
joint_summary = survey_summary.merge(
    simulation_summary,
    on=["condition", "condition_label", "policy", "pov", "axis"],
    how="outer",
)

joint_summary["mean_gap_sim_minus_survey"] = joint_summary["sim_mean_0_1"] - joint_summary["survey_mean_0_1"]

display(
    joint_summary[
        [
            "condition",
            "condition_label",
            "axis",
            "survey_mean_raw",
            "survey_mean_0_1",
            "sim_mean_0_1",
            "mean_gap_sim_minus_survey",
            "n",
            "n_episode_slots",
            "n_logged_runs",
            "technical_valid_rate",
        ]
    ].sort_values(["axis", "condition"])
)

In [ ]:
plot_data = joint_summary.dropna(subset=["survey_mean_0_1", "sim_mean_0_1"]).copy()

fig, ax = plt.subplots(figsize=(7, 7))
for axis_name, axis_frame in plot_data.groupby("axis"):
    ax.scatter(axis_frame["survey_mean_0_1"], axis_frame["sim_mean_0_1"], label=axis_name, s=70)

ax.axline((0, 0), slope=1, color="black", linewidth=1, linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Survey mean, normalized to 0..1")
ax.set_ylabel("Simulation mean, 0..1")
ax.set_title("Absolute score check: survey vs simulation")
ax.legend(title="Axis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

The scatter above is a calibration check, not the main decision. If the simulation scores are all near `1.0` while the survey scores are around neutral or negative, the benchmark may still rank policies correctly, but its absolute score scale is not calibrated to human perception.

## Paired Difference Analysis

This is the core mixed analysis. For the survey, each difference is computed within participant. For the simulation, each difference is computed within episode slot. Then we compare the direction of those differences.

We use small practical deadbands so tiny numerical differences are treated as ties:

- Survey normalized delta threshold: `0.03`, which is about `0.18` points on the original `-3..+3` scale.
- Simulation delta threshold: `0.02` on the `0..1` benchmark scale.

These thresholds are not magic. They are there to avoid overinterpreting tiny differences.

In [ ]:
SURVEY_DELTA_THRESHOLD = 0.03
SIMULATION_DELTA_THRESHOLD = 0.02


def percentile_bootstrap_ci(values, statistic="mean", n_resamples=5000, confidence=0.95, seed=RANDOM_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return float(values[0]), float(values[0])

    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    samples = values[sample_indices]
    if statistic == "median":
        bootstrap_statistics = np.median(samples, axis=1)
    else:
        bootstrap_statistics = np.mean(samples, axis=1)

    alpha = 1 - confidence
    return tuple(np.quantile(bootstrap_statistics, [alpha / 2, 1 - alpha / 2]))


def classify_direction(value, threshold):
    if pd.isna(value):
        return "missing"
    if value > threshold:
        return "left_higher"
    if value < -threshold:
        return "right_higher"
    return "tie"


def compare_directions(survey_direction, simulation_direction):
    if "missing" in {survey_direction, simulation_direction}:
        return "missing"
    if survey_direction == simulation_direction:
        return "agree"
    if "tie" in {survey_direction, simulation_direction}:
        return "one_tie"
    return "disagree"


def advantage_from_deltas(deltas, threshold):
    deltas = np.asarray(deltas, dtype=float)
    deltas = deltas[~np.isnan(deltas)]
    if len(deltas) == 0:
        return np.nan, 0, 0, 0
    wins = int((deltas > threshold).sum())
    ties = int((np.abs(deltas) <= threshold).sum())
    losses = int((deltas < -threshold).sum())
    relative_advantage = (wins + 0.5 * ties) / len(deltas)
    return relative_advantage, wins, ties, losses

In [ ]:
comparison_rows = []

for left_condition, right_condition, comparison_name, meaning in COMPARISONS:
    for axis in AXES_WITH_GLOBAL:
        left_survey = (
            survey_analysis_long[(survey_analysis_long["condition"] == left_condition) & (survey_analysis_long["axis"] == axis)]
            .set_index("participant_id")["score_norm"]
            .rename("left")
        )
        right_survey = (
            survey_analysis_long[(survey_analysis_long["condition"] == right_condition) & (survey_analysis_long["axis"] == axis)]
            .set_index("participant_id")["score_norm"]
            .rename("right")
        )
        survey_pair = pd.concat([left_survey, right_survey], axis=1, join="inner").dropna()
        survey_deltas = survey_pair["left"] - survey_pair["right"]
        survey_ci_low, survey_ci_high = percentile_bootstrap_ci(survey_deltas, statistic="mean")
        survey_advantage, survey_wins, survey_ties, survey_losses = advantage_from_deltas(survey_deltas, SURVEY_DELTA_THRESHOLD)

        left_sim = (
            simulation_analysis_long[(simulation_analysis_long["condition"] == left_condition) & (simulation_analysis_long["axis"] == axis)]
            .set_index("episode_slot")["sim_score"]
            .rename("left")
        )
        right_sim = (
            simulation_analysis_long[(simulation_analysis_long["condition"] == right_condition) & (simulation_analysis_long["axis"] == axis)]
            .set_index("episode_slot")["sim_score"]
            .rename("right")
        )
        simulation_pair = pd.concat([left_sim, right_sim], axis=1, join="inner").dropna()
        simulation_deltas = simulation_pair["left"] - simulation_pair["right"]
        simulation_advantage, simulation_wins, simulation_ties, simulation_losses = advantage_from_deltas(simulation_deltas, SIMULATION_DELTA_THRESHOLD)

        survey_mean_delta = survey_deltas.mean()
        simulation_mean_delta = simulation_deltas.mean()
        survey_direction = classify_direction(survey_mean_delta, SURVEY_DELTA_THRESHOLD)
        simulation_direction = classify_direction(simulation_mean_delta, SIMULATION_DELTA_THRESHOLD)

        comparison_rows.append(
            {
                "comparison": comparison_name,
                "meaning": meaning,
                "axis": axis,
                "left": left_condition,
                "right": right_condition,
                "survey_n_participants": len(survey_deltas),
                "survey_mean_delta_norm": survey_mean_delta,
                "survey_mean_delta_ci_low": survey_ci_low,
                "survey_mean_delta_ci_high": survey_ci_high,
                "survey_direction": survey_direction,
                "survey_ci_excludes_zero": (survey_ci_low > 0) or (survey_ci_high < 0),
                "survey_relative_advantage": survey_advantage,
                "survey_wins": survey_wins,
                "survey_ties": survey_ties,
                "survey_losses": survey_losses,
                "simulation_n_episode_pairs": len(simulation_deltas),
                "simulation_mean_delta": simulation_mean_delta,
                "simulation_direction": simulation_direction,
                "simulation_relative_advantage": simulation_advantage,
                "simulation_wins": simulation_wins,
                "simulation_ties": simulation_ties,
                "simulation_losses": simulation_losses,
                "direction_agreement": compare_directions(survey_direction, simulation_direction),
            }
        )

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.sort_values(["axis", "comparison"]))

In [ ]:
global_comparisons = comparison_results[comparison_results["axis"] == "global"].copy()
positions = np.arange(len(global_comparisons))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(positions - 0.18, global_comparisons["survey_mean_delta_norm"], height=0.35, label="Survey delta")
ax.barh(positions + 0.18, global_comparisons["simulation_mean_delta"], height=0.35, label="Simulation delta")
ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels(global_comparisons["comparison"])
ax.set_xlabel("Left minus right difference")
ax.set_title("Global paired differences: survey vs simulation")
ax.legend()
plt.tight_layout()
plt.show()

display(global_comparisons[["comparison", "survey_mean_delta_norm", "survey_mean_delta_ci_low", "survey_mean_delta_ci_high", "simulation_mean_delta", "direction_agreement"]])

## Ranking Alignment

This section checks whether the four conditions are ordered similarly by the survey and the simulation. With only four conditions per axis, the rank correlation is descriptive. It is useful as a compact sanity check, not as a final proof.

In [ ]:
def rank_correlation(left_values, right_values):
    data = pd.DataFrame({"left": left_values, "right": right_values}).dropna()
    if len(data) < 2:
        return np.nan
    left_ranks = data["left"].rank()
    right_ranks = data["right"].rank()
    if left_ranks.nunique() < 2 or right_ranks.nunique() < 2:
        return np.nan
    return left_ranks.corr(right_ranks)


rank_rows = []
for axis in AXES_WITH_GLOBAL:
    axis_frame = joint_summary[joint_summary["axis"] == axis].dropna(subset=["survey_mean_0_1", "sim_mean_0_1"]).copy()
    rank_rows.append(
        {
            "axis": axis,
            "n_conditions": len(axis_frame),
            "rank_correlation": rank_correlation(axis_frame["survey_mean_0_1"], axis_frame["sim_mean_0_1"]),
            "survey_order_high_to_low": " > ".join(axis_frame.sort_values("survey_mean_0_1", ascending=False)["condition"].str.upper()),
            "simulation_order_high_to_low": " > ".join(axis_frame.sort_values("sim_mean_0_1", ascending=False)["condition"].str.upper()),
        }
    )

rank_alignment = pd.DataFrame(rank_rows)
overall_rank_correlation = rank_correlation(plot_data["survey_mean_0_1"], plot_data["sim_mean_0_1"])

display(rank_alignment)
print(f"Overall descriptive rank correlation across available condition-axis points: {overall_rank_correlation:.3f}")

## Did We Do It Right Or Wrong?

This should be read as a benchmark validation check, not as a moral yes/no. The benchmark is working for this dataset when it predicts the same relative outcomes as the survey. It is not working, or is incomplete, where the survey and simulation disagree in direction or where the simulation has too little evidence.

In [ ]:
agreement_summary = (
    comparison_results.groupby(["axis", "direction_agreement"])
    .size()
    .unstack(fill_value=0)
    .reindex(AXES_WITH_GLOBAL)
)

decisive_results = comparison_results[comparison_results["direction_agreement"] != "missing"].copy()
agreement_rate = (decisive_results["direction_agreement"] == "agree").mean()

mismatches = comparison_results[comparison_results["direction_agreement"].isin(["disagree", "one_tie", "missing"])].copy()

display(agreement_summary)
print(f"Agreement rate across comparison rows: {agreement_rate:.1%}")

display(
    mismatches[
        [
            "comparison",
            "axis",
            "survey_mean_delta_norm",
            "survey_mean_delta_ci_low",
            "survey_mean_delta_ci_high",
            "survey_direction",
            "simulation_mean_delta",
            "simulation_direction",
            "direction_agreement",
        ]
    ].sort_values(["axis", "comparison"])
)

### How To Interpret The Output

A useful benchmark should usually satisfy these checks:

- The survey paired differences are stable enough to interpret, ideally with bootstrap intervals that do not cross zero for the main comparisons.
- The simulation deltas have the same direction as the survey deltas for the same policy/POV comparison.
- The rank order of Q1, Q2, Q3, and Q4 is similar between survey and simulation.
- The simulation has valid runs for every condition and episode slot.

Important limits:

- The current survey order was fixed, so order effects are possible.
- The simulation side has episode/run variability, not participant variability.
- If the simulation scores are saturated near `1.0`, the benchmark may not be calibrated even when it ranks correctly.
- With only three episode slots, simulation-side statistical claims should stay descriptive.

So the practical conclusion is: if the direction agreement and rank alignment are high, the benchmark is directionally useful. If they are low, the metric model or weighting is probably not matching human perception yet.